# BP2 Gate 5 — Decision Layer & Reporting
**Customer360 Navigator Enterprise Suite — Customer Friction Classification**

## Purpose
Turns Gate 4's validated champion into a per-row decision-record layer: a prediction, a
confidence score, and a grounded reason code for every real held-out test row — the same
governance step BP1 Gate 5 built, reused here (HYPER) with BP2's structured feature space and
4-class severity taxonomy in place of BP1's TF-IDF text and 77-class intent.

## Scope decision carried over from BP1 (same standing decision, not re-litigated here)
This gate is a fully offline, deterministic decision-record layer — **no GenAI API call**. Real
GenAI-drafted customer-facing text (and the UDAAP/NIST AI RMF review that requires) is scoped to
BP6 per the Master Execution Plan, the identical scope decision the user confirmed for BP1 Gate 5.

## What this notebook does
1. Reads the real champion **live** from Gate 4's `gate4_statistical_validation.json`, cross-checks
   it against Gate 3's recorded champion — never hardcoded here.
2. Rebuilds Gate 3/4's exact feature engineering (Sections 5-7, reused verbatim) and refits the
   champion on the full real train split.
3. Predicts on the full real held-out test set (not a sample) — every row gets a decision record.
4. Recomputes overall test accuracy and cross-checks it against Gate 3's recorded value.
5. Computes per-instance SHAP on a bounded 150-row sample (laptop-safe), producing grounded reason
   codes: for the shared one-hot/frequency feature space, a reason code is only ever drawn from a
   feature that is literally nonzero in that row (the category genuinely present in that
   complaint's own data); for CatBoost's raw categorical path — no "absent" concept, since every
   column always holds a real value — reason codes are formatted as `Column=Value` so each stays
   self-descriptive without needing that masking.
6. Cross-checks its own sample-aggregated top reason-code terms against Gate 4's global top-10.

## Honest limitation on the top-3 confidence breakdown
With only 4 real severity classes (vs BP1's 77 intents), "top-3 of 4" already covers most of the
class distribution — rank-3 is nearly always just the lowest-probability remaining class, not a
meaningfully distinct alternative the way it was in BP1's 77-class case. Reported anyway for
structural consistency with BP1, but this limitation is stated plainly rather than presented as
more informative than it is.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; it does not run it. Every
  number is computed live during the real run.
- **HYPER**: Gate 3/4's feature-engineering code (Gold-layer reload, train/test split, shared
  one-hot/frequency preprocessing, CatBoost raw-categorical path) and Gate 4's SHAP densification
  fix (always densify for TreeExplainer unless the champion is LogisticRegression, regardless of
  Gate 3's fit-time-only `NEEDS_DENSE` set) are both reused verbatim. `src/utils/bp1_config_sync.py`
  reused unmodified.
- **Continue gracefully on failure**: SHAP computation is wrapped in try/except — a failure is
  recorded plainly (`shap_error`) and decision records still get written with empty reason codes
  rather than the whole gate halting.
- **Idempotent**: re-running overwrites this gate's artifacts and its own `gate5_...` config block,
  without touching Gates 1-4's fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp2_customer_friction_classification/artifacts/gate5_decision_records.csv` (one row
  per real held-out test example)
- `notebooks/bp2_customer_friction_classification/artifacts/gate5_decision_layer_summary.json`
- `notebooks/bp2_customer_friction_classification/artifacts/model_inventory_entry.json` (Gate 5
  fields added to the existing Gate 3/4 entry, in place)
- `configs/bp2_customer_friction_classification.yaml` — `gate5_decision_layer` block appended/updated

## Prerequisites
BP2 Gates 3 and 4 must both have been real-run (this notebook reads Gate 4's confirmed champion
and raises if that file is missing). `shap` must be installed — same live check Gate 4 performs.

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the accuracy-consistency check fails, this
notebook's feature-engineering code has drifted from Gate 3/4's — fix the drift, do not silence it.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP2 Gate 5 decision layer / reporting notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402
from catboost import CatBoostClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP2 Gate 5 and was not confirmed installed. "
        "Run `pip install shap` before running this notebook."
    )
import shap  # noqa: E402
print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp2_customer_friction_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_friction_severity_gold.parquet"
BP2_CONFIG_PATH = CONFIGS_DIR / "bp2_customer_friction_classification.yaml"
SEVERITY_CONFIG_PATH = CONFIGS_DIR / "bp2_friction_severity_taxonomy.yaml"

for p in (GOLD_PATH, BP2_CONFIG_PATH, SEVERITY_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Required input not found: {p}. Confirm BP2 Gates 1-4 all completed for real.")

# ============================================================
# SECTION 4: Load Gate 3/4's real results - champion read LIVE, never hardcoded
# ============================================================
with open(BP2_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp2_config = yaml.safe_load(f)
with open(SEVERITY_CONFIG_PATH, "r", encoding="utf-8") as f:
    severity_config = yaml.safe_load(f)

TARGET_COL = bp2_config["target_definition"]["primary_target"]
RANDOM_STATE = bp2_config["random_state"]
ORDINAL_CLASSES = list(severity_config["severity_classes"].keys())
N_CLASSES = len(ORDINAL_CLASSES)

gate3_block = bp2_config.get("gate3_model_benchmark")
assert gate3_block is not None, "[CHECK FAILED] gate3_model_benchmark missing - run BP2 Gate 3 first."

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), (
    f"[CHECK FAILED] {gate4_json_path} not found - run BP2 Gate 4 first (this gate reads its confirmed champion)."
)
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)
CHAMPION_NAME = gate4_results["champion_model"]
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 4 recorded '{CHAMPION_NAME}' but Gate 3's config block says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3/4."
)
gate3_recorded_test_accuracy = float(gate3_block["held_out_test_accuracy"])
print(f"[OK] Champion (live, re-verified against Gate 3 + Gate 4): {CHAMPION_NAME}")

cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame EXACTLY as Gates 3/4 did (HYPER reuse) -
# this gate's refit/predict is only meaningful on the IDENTICAL rows/columns Gate 3 benchmarked.
# ============================================================
FEATURE_COLS_CATEGORICAL = ["Product", "Sub-product", "Issue", "Sub-issue", "State",
                            "Submitted via", "common_taxonomy_bucket"]
COMPANY_COL = "Company"
BARRED_COLUMNS = ["Company response to consumer", "Timely response?", "Date received",
                  "Date sent to company", "Company public response", "Complaint ID", "ZIP code", "Tags"]

gold_lazy = pl.scan_parquet(GOLD_PATH)
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL]
df_pl = (
    gold_lazy.select(select_cols)
    .filter(pl.col(TARGET_COL).is_in(ORDINAL_CLASSES))
    .collect()
)
for barred in BARRED_COLUMNS:
    assert barred not in df_pl.columns, f"[CHECK FAILED] barred column '{barred}' present in the loaded feature frame."
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,}.")

feature_data = {col: df_pl[col].cast(pl.Utf8).fill_null("MISSING").to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Utf8).to_list()
X_full = pd.DataFrame(feature_data)
y_full_labels = pd.Series(target_data, name=TARGET_COL)

# ============================================================
# SECTION 6: Identical stratified train/test split as Gates 3/4
# ============================================================
X_train_raw, X_test_raw, y_train_labels, y_test_labels = train_test_split(
    X_full, y_full_labels, test_size=0.20, stratify=y_full_labels, random_state=RANDOM_STATE
)
label_encoder = LabelEncoder().fit(y_train_labels)
y_train = label_encoder.transform(y_train_labels)
y_test = label_encoder.transform(y_test_labels)
CLASS_NAMES = label_encoder.classes_
print(f"[OK] Reproduced Gate 3/4's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

# ============================================================
# SECTION 7: Rebuild the shared preprocessing EXACTLY as Gates 3/4 (fit on TRAIN only)
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
test_company_freq = X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")

CATBOOST_FEATURE_COLS = FEATURE_COLS_CATEGORICAL + [COMPANY_COL]
X_train_cat = X_train_raw[CATBOOST_FEATURE_COLS].copy()
X_test_cat = X_test_raw[CATBOOST_FEATURE_COLS].copy()
CATBOOST_CAT_FEATURE_INDICES = list(range(len(CATBOOST_FEATURE_COLS)))
SHARED_FEATURE_NAMES = np.array(list(ohe.get_feature_names_out()) + ["Company_freq"])

CANDIDATES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, class_weight="balanced", n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=1, verbose=-1,
        random_state=cv_settings["random_state"],
    ),
    "catboost": CatBoostClassifier(
        iterations=100, thread_count=1, verbose=False, allow_writing_files=False,
        auto_class_weights="Balanced", random_state=cv_settings["random_state"],
        cat_features=CATBOOST_CAT_FEATURE_INDICES,
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}
USES_RAW_CATEGORICAL = {"catboost"}

# ============================================================
# SECTION 8: Refit champion on FULL train, predict on the FULL held-out test set
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
champion_model = CANDIDATES[CHAMPION_NAME]
if CHAMPION_NAME in USES_RAW_CATEGORICAL:
    X_train_final, X_test_final = X_train_cat, X_test_cat
    y_train_final = y_train_labels.to_numpy()
else:
    X_train_final, X_test_final = X_train_shared, X_test_shared
    if CHAMPION_NAME in NEEDS_DENSE:
        X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
        X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)
    y_train_final = y_train

print(f"\n[GATE5] Refitting champion ({CHAMPION_NAME}) on the full train split...")
t0 = time.perf_counter()
champion_model.fit(X_train_final, y_train_final)
print(f"[GATE5] Fit done in {time.perf_counter() - t0:.1f}s")

assert hasattr(champion_model, "predict_proba"), (
    f"[CHECK FAILED] Champion {CHAMPION_NAME} has no predict_proba - Gate 5's confidence score requires it."
)
if CHAMPION_NAME in USES_RAW_CATEGORICAL:
    y_pred_raw = champion_model.predict(X_test_final)
    y_pred_encoded = label_encoder.transform(np.asarray(y_pred_raw).ravel())
    y_proba_raw = champion_model.predict_proba(X_test_final)
    proba_classes = list(champion_model.classes_)
    proba_order = [proba_classes.index(c) for c in label_encoder.classes_]
    y_proba = np.asarray(y_proba_raw)[:, proba_order]
else:
    y_pred_encoded = np.asarray(champion_model.predict(X_test_final)).astype(int).ravel()
    y_proba = champion_model.predict_proba(X_test_final)
    pipeline_classes = list(champion_model.classes_)
    assert pipeline_classes == list(range(N_CLASSES)), (
        f"[CHECK FAILED] Champion's class order does not match the expected 0..{N_CLASSES - 1} "
        "integer-encoded order - probability-column alignment would be wrong."
    )

proba_row_sums = y_proba.sum(axis=1)
assert np.allclose(proba_row_sums, 1.0, atol=1e-6), (
    "[CHECK FAILED] predict_proba rows do not sum to 1.0 - something is wrong with the champion's probability output."
)

overall_accuracy = float(accuracy_score(y_test, y_pred_encoded))
accuracy_consistency_diff = abs(overall_accuracy - gate3_recorded_test_accuracy)
print(f"[CHECK] Recomputed overall test accuracy: {overall_accuracy:.6f} "
      f"(Gate 3 recorded: {gate3_recorded_test_accuracy:.6f}, diff={accuracy_consistency_diff:.6f})")

# Top-3 predictions per row - with only 4 real severity classes, "top-3 of 4" already covers most of
# the distribution (rank3 is nearly always the lowest-probability remaining class), stated plainly
# rather than presented as more differentiating than it is; still a real, useful confidence signal.
top3_idx = np.argsort(y_proba, axis=1)[:, ::-1][:, :3]
top1_conf = y_proba[np.arange(len(y_proba)), top3_idx[:, 0]]
rank2_label_idx = top3_idx[:, 1]
rank2_conf = y_proba[np.arange(len(y_proba)), rank2_label_idx]
rank3_label_idx = top3_idx[:, 2]
rank3_conf = y_proba[np.arange(len(y_proba)), rank3_label_idx]

mean_conf_correct = float(top1_conf[y_pred_encoded == y_test].mean())
mean_conf_incorrect = float(top1_conf[y_pred_encoded != y_test].mean()) if (y_pred_encoded != y_test).any() else None
print(f"[RESULT] Mean top-1 confidence - correct predictions: {mean_conf_correct:.4f}, "
      f"incorrect predictions: {mean_conf_incorrect if mean_conf_incorrect is None else round(mean_conf_incorrect, 4)}")

# ============================================================
# SECTION 9: Per-instance SHAP - explainer chosen by champion's model type (mirrors Gate 4's
# densification logic exactly), bounded sample for laptop safety, reason codes grounded by
# construction: for the shared one-hot/frequency features, only nonzero-valued features in that
# row are ever reported (the categories literally present in that row, same grounding principle
# BP1 used for nonzero TF-IDF weight); for CatBoost's raw categorical path every column always has
# a real value by definition (no "absent" concept), so reason codes there are the top-ranked
# columns formatted as "Column=Value" so each is self-descriptive on its own, same standard the
# one-hot column names already provide "for free" via their own naming.
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test_raw))
SHAP_BACKGROUND_SIZE = min(50, len(X_train_raw))
N_REASON_CODES = 5
shap_error = None
sample_idx = np.array([], dtype=int)
reason_codes_by_row = {}
grounding_failures = 0
is_linear_champion = isinstance(champion_model, LogisticRegression)
try:
    shap_rng = np.random.RandomState(cv_settings["random_state"])
    sample_idx = shap_rng.choice(len(X_test_raw), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = shap_rng.choice(len(X_train_raw), size=SHAP_BACKGROUND_SIZE, replace=False)

    if CHAMPION_NAME in USES_RAW_CATEGORICAL:
        X_sample = X_test_cat.iloc[sample_idx]
        X_bg = X_train_cat.iloc[bg_idx]
        feature_names = np.array(CATBOOST_FEATURE_COLS)
    else:
        X_sample_vec = X_test_shared[sample_idx]
        X_bg_vec = X_train_shared[bg_idx]
        feature_names = SHARED_FEATURE_NAMES
        X_sample_dense_for_masking = np.asarray(X_sample_vec.todense())
        if not is_linear_champion:
            X_sample_vec = np.asarray(X_sample_vec.todense(), dtype=np.float32)
            X_bg_vec = np.asarray(X_bg_vec.todense(), dtype=np.float32)
        X_sample, X_bg = X_sample_vec, X_bg_vec

    if is_linear_champion:
        print(f"\n[GATE5] SHAP: using LinearExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)...")
        explainer = shap.LinearExplainer(champion_model, X_bg)
        shap_values = explainer.shap_values(X_sample)
    else:
        print(f"\n[GATE5] SHAP: using TreeExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)...")
        explainer = shap.TreeExplainer(champion_model)
        shap_values = explainer.shap_values(X_sample)

    sample_pred_idx = y_pred_encoded[sample_idx]
    if isinstance(shap_values, list):
        per_row_shap = np.stack(
            [np.asarray(shap_values[cls])[i] for i, cls in enumerate(sample_pred_idx)], axis=0
        )
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            if arr.shape[-1] == N_CLASSES:
                per_row_shap = arr[np.arange(len(sample_idx)), :, sample_pred_idx]
            else:
                per_row_shap = arr[np.arange(len(sample_idx)), sample_pred_idx, :]
        else:
            per_row_shap = arr

    assert per_row_shap.shape == (len(sample_idx), len(feature_names)), (
        f"[CHECK FAILED] Per-row SHAP shape {per_row_shap.shape} does not match "
        f"(sample_size={len(sample_idx)}, feature_count={len(feature_names)})."
    )

    for local_i, global_row in enumerate(sample_idx):
        row_shap = per_row_shap[local_i]
        if CHAMPION_NAME in USES_RAW_CATEGORICAL:
            # Every raw categorical column always has a real value (MISSING is itself a valid,
            # real category) - no "absent" concept, so every feature is eligible; formatted as
            # "Column=Value" so each code is self-descriptive without needing the masking below.
            top_k = min(N_REASON_CODES, len(feature_names))
            top_feat_idx = np.argsort(np.abs(row_shap))[::-1][:top_k]
            row_values = X_test_cat.iloc[global_row]
            codes = [f"{feature_names[j]}={row_values[feature_names[j]]}" for j in top_feat_idx]
            reason_codes_by_row[global_row] = codes
        else:
            row_nonzero_mask = X_sample_dense_for_masking[local_i] != 0.0
            masked_shap = np.where(row_nonzero_mask, np.abs(row_shap), -np.inf)
            if not row_nonzero_mask.any():
                reason_codes_by_row[global_row] = []
                continue
            top_k = min(N_REASON_CODES, int(row_nonzero_mask.sum()))
            top_feat_idx = np.argsort(masked_shap)[::-1][:top_k]
            codes = [str(feature_names[j]) for j in top_feat_idx]
            for j in top_feat_idx:
                if not row_nonzero_mask[j]:
                    grounding_failures += 1
            reason_codes_by_row[global_row] = codes

    all_codes = [c for codes in reason_codes_by_row.values() for c in codes]
    gate5_top_terms = pd.Series(all_codes).value_counts().head(10).index.tolist() if all_codes else []
    print(f"[RESULT] Gate 5's own independently-aggregated top reason-code terms (by frequency, sampled): "
          f"{gate5_top_terms}")

except Exception as e:  # noqa: BLE001 - continue gracefully; decision records still get written
    shap_error = f"{type(e).__name__}: {e}"
    gate5_top_terms = []
    print(f"[LIMITATION] Per-instance SHAP failed for champion model family "
          f"'{type(CANDIDATES[CHAMPION_NAME]).__name__}': {shap_error}. Decision records below will still "
          "be written with predicted labels/confidence, but with empty reason_codes.")

# ============================================================
# SECTION 10: Cross-check against Gate 4's already-saved global top-10 SHAP features
# ============================================================
gate4_shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
gate4_top_terms = []
if gate4_shap_csv_path.exists():
    gate4_shap_df = pd.read_csv(gate4_shap_csv_path)
    if len(gate4_shap_df) > 0:
        gate4_top_terms = gate4_shap_df["feature"].head(10).tolist()
overlap_terms = sorted(set(gate5_top_terms) & set(gate4_top_terms))
print(f"\n[CHECK] Gate 4 global top-10 vs Gate 5 sample-aggregated top-10 overlap: {len(overlap_terms)} terms "
      f"in common ({overlap_terms}). A raw-categorical champion's 'Column=Value' codes will not literally "
      "match Gate 4's bare feature names even on real overlap - this check is most informative when the "
      "champion uses the shared one-hot feature space.")

# ============================================================
# SECTION 11: Assemble decision records for the FULL held-out test set (one row per test example)
# ============================================================
in_sample_set = set(int(i) for i in sample_idx)
records = []
for i in range(len(X_test_raw)):
    codes = reason_codes_by_row.get(i, None) if i in in_sample_set else None
    row_summary = "; ".join(f"{col}={X_test_raw.iloc[i][col]}" for col in FEATURE_COLS_CATEGORICAL + [COMPANY_COL])
    records.append({
        "row_index": i,
        "true_label": str(y_test_labels.iloc[i]),
        "predicted_label": str(CLASS_NAMES[y_pred_encoded[i]]),
        "correct": bool(y_pred_encoded[i] == y_test[i]),
        "confidence_top1": round(float(top1_conf[i]), 4),
        "rank2_label": str(CLASS_NAMES[rank2_label_idx[i]]),
        "rank2_confidence": round(float(rank2_conf[i]), 4),
        "rank3_label": str(CLASS_NAMES[rank3_label_idx[i]]),
        "rank3_confidence": round(float(rank3_conf[i]), 4),
        "in_shap_sample": i in in_sample_set,
        "reason_codes": "|".join(codes) if codes else "",
        "feature_summary": row_summary,
    })
decision_records_df = pd.DataFrame(records)
assert len(decision_records_df) == len(X_test_raw), (
    f"[CHECK FAILED] Decision-record count ({len(decision_records_df)}) does not match test-set size ({len(X_test_raw)})."
)

records_path = ARTIFACTS_DIR / "gate5_decision_records.csv"
decision_records_df.to_csv(records_path, index=False)
print(f"\n[SAVED] {records_path.relative_to(PROJECT_ROOT)} ({len(decision_records_df):,} decision records, "
      f"{len(in_sample_set):,} with reason codes)")

# ============================================================
# SECTION 12: Write summary (idempotent overwrite-in-place)
# ============================================================
summary = {
    "bp_id": "bp2",
    "gate": 5,
    "champion_model": CHAMPION_NAME,
    "n_decision_records": int(len(decision_records_df)),
    "n_with_reason_codes": int(len(in_sample_set)),
    "shap_sample_size_bound": SHAP_SAMPLE_SIZE,
    "n_reason_codes_per_record": N_REASON_CODES,
    "shap_error": shap_error,
    "overall_test_accuracy_recomputed": round(overall_accuracy, 6),
    "gate3_recorded_test_accuracy": round(gate3_recorded_test_accuracy, 6),
    "accuracy_consistency_diff": round(accuracy_consistency_diff, 6),
    "mean_confidence_correct_predictions": round(mean_conf_correct, 4),
    "mean_confidence_incorrect_predictions": round(mean_conf_incorrect, 4) if mean_conf_incorrect is not None else None,
    "gate5_aggregated_top_reason_code_terms": gate5_top_terms,
    "gate4_global_top10_terms": gate4_top_terms,
    "overlap_terms_with_gate4": overlap_terms,
    "overlap_count_with_gate4": len(overlap_terms),
    "reason_code_grounding_failures": int(grounding_failures),
    "reason_code_grounding_method": (
        "For the shared one-hot/frequency feature space, a reason code is only ever reported for a row if "
        "that feature's value is nonzero in that row - i.e. the category is literally present in that row "
        "by construction of the one-hot encoding, not asserted after the fact. For CatBoost's raw "
        "categorical path, every column always has a real value (MISSING is itself a valid category), so "
        "reason codes there are formatted as 'Column=Value' to stay equally self-descriptive."
    ),
    "compliance_touchpoint": {
        "udaap_language_review": (
            "Not Applicable to BP2 Gate 5 - this gate generates no GenAI or customer-facing text; reason "
            "codes and confidence scores are deterministic outputs of the champion classifier and real "
            "per-instance SHAP values. Real GenAI-drafted customer-facing text (subject to UDAAP review) is "
            "scoped to BP6 (GenAI Resolution Assistant) per the Master Execution Plan, same standing scope "
            "decision as BP1 Gate 5."
        ),
        "nist_ai_rmf_measure_manage": (
            "Not Applicable to BP2 Gate 5 for the same reason - no GenAI output is produced here. Applies at BP6."
        ),
        "genai_api_used": False,
        "scope_decision_confirmed_by_user_utc": "2026-09-22",
    },
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"[SAVED] {summary_path.relative_to(PROJECT_ROOT)}")

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp2", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 5 decision layer + reporting complete"
model_inventory_entry["gate5_n_decision_records"] = int(len(decision_records_df))
model_inventory_entry["gate5_overall_test_accuracy"] = round(overall_accuracy, 6)
model_inventory_entry["gate5_genai_api_used"] = False
model_inventory_entry["gate5_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 5 fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP2_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = current_status_line.split('"')[1] + "_gate5_confirmed" \
    if "_gate5_confirmed" not in current_status_line else current_status_line.split('"')[1]
status_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE)
BP2_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate5_marker = "# --- Gate 5 (Decision Layer & Reporting) results (appended, idempotent overwrite) ---"
gate5_block_lines = [
    "gate5_decision_layer:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  n_decision_records: {int(len(decision_records_df))}",
    f"  n_with_reason_codes: {int(len(in_sample_set))}",
    f"  overall_test_accuracy_recomputed: {round(overall_accuracy, 6)}",
    "  genai_api_used: false",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP2_CONFIG_PATH, gate5_marker, gate5_block_lines)
print(f"[SAVED] {BP2_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate5_decision_layer block)")

# ============================================================
# SECTION 13: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate3_and_gate4_recorded": CHAMPION_NAME == gate3_block["champion_model"],
    "decision_records_count_equals_test_set_size": len(decision_records_df) == len(X_test_raw),
    "all_records_have_predicted_label_and_confidence": decision_records_df["predicted_label"].notna().all()
        and decision_records_df["confidence_top1"].notna().all(),
    "confidence_scores_within_valid_range": decision_records_df["confidence_top1"].between(0.0, 1.0).all(),
    "probabilities_sum_to_one_per_row": bool(np.allclose(proba_row_sums, 1.0, atol=1e-6)),
    "shap_sample_size_matches_configured_bound": len(sample_idx) == SHAP_SAMPLE_SIZE or shap_error is not None,
    "all_reason_codes_grounded_by_nonzero_value_or_raw_categorical": grounding_failures == 0,
    "accuracy_consistency_with_gate3_recorded": accuracy_consistency_diff < 1e-4,
    "no_barred_column_in_feature_frame": all(b not in df_pl.columns for b in BARRED_COLUMNS),
    "compliance_touchpoint_documented": "compliance_touchpoint" in summary and bool(summary["compliance_touchpoint"]),
    "decision_records_csv_written": records_path.exists(),
    "summary_json_written": summary_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp2_config_yaml_updated": BP2_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP2 Gate 5 complete. {len(decision_records_df):,} decision records written "
      f"({len(in_sample_set):,} with grounded reason codes). Recomputed test accuracy={round(overall_accuracy, 4)} "
      f"(Gate 3 recorded: {round(gate3_recorded_test_accuracy, 4)}). "
      f"Gate 4/Gate 5 SHAP top-term overlap: {len(overlap_terms)}/10. "
      "No GenAI API used (offline decision-record layer, same scope decision as BP1). "
      "Proceed to BP2 Gate 6 (Productization, Monitoring & Governance) next.")
